# Notebook 5 – Logistic Regression

Topics:
- Classification
- Binary Classification
- Logistic Regression
- Sigmoid Function
- Probability
- Decision Boundary
- Threshold
- Log Loss
- Coefficients
- Multiclass Classification

## 1. Classification

Classification is a supervised machine learning technique used to predict categories or classes.

Examples:
- Spam / Not Spam
- Pass / Fail
- Churn / Not Churn
- Disease / No Disease

## 2. Binary Classification

Binary classification has exactly two possible classes.

Example:
- 0 = No Churn
- 1 = Churn

Other examples:
- 0 = Not Spam, 1 = Spam
- 0 = Fail, 1 = Pass

## 3. Logistic Regression

Logistic Regression is a supervised classification algorithm.

It first calculates a linear equation:

z = b + w1*x1 + w2*x2 + ...

Then it passes the result through the sigmoid function to produce a probability between 0 and 1.

The probability is then converted into a class using a threshold.

## 4. Why Logistic Regression is a Classification Algorithm

Despite the word Regression in its name, Logistic Regression is mainly used for classification.

Linear Regression predicts continuous numerical values.

Logistic Regression predicts the probability of a class and then converts that probability into a class.

Workflow:

Features → Linear Equation → Sigmoid → Probability → Threshold → Class

## 5. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss

## 6. Load Dataset

We will use the Breast Cancer dataset from Scikit-learn.

Target classes:
- 0 = Malignant
- 1 = Benign

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="Target")

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Target classes:", np.unique(y))

print("\nTarget meaning:")
print("0 = Malignant")
print("1 = Benign")

## 7. Inspect the Dataset

In [ ]:
print("First 5 rows:")
print(X.head())

print("\nDataset shape:", X.shape)

print("\nClass distribution:")
print(y.value_counts())

## 8. Train/Test Split

Training data is used to train the model.

Testing data is used to evaluate the model on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 9. Feature Scaling

Logistic Regression generally works better when numerical features are on similar scales.

The scaler is fitted only on the training data to prevent data leakage.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")

## 10. Train Logistic Regression Model

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)

model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully.")

## 11. Prediction

`predict()` returns the predicted class.

`predict_proba()` returns the probability of each class.

In [ ]:
y_pred = model.predict(X_test_scaled)
y_probability = model.predict_proba(X_test_scaled)

print("First 10 predicted classes:")
print(y_pred[:10])

print("\nFirst 10 actual classes:")
print(y_test.iloc[:10].values)

## 12. Probability

For binary classification, `predict_proba()` returns two probabilities for each sample:

- Probability of Class 0
- Probability of Class 1

The two probabilities add up to 1.

In [ ]:
probability_df = pd.DataFrame(
    y_probability,
    columns=["Probability_Class_0", "Probability_Class_1"]
)

print(probability_df.head(10))

## 13. Sigmoid Function

The sigmoid function converts a value into a probability between 0 and 1.

Formula:

sigmoid(z) = 1 / (1 + e^(-z))

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_values = np.array([-5, -3, -1, 0, 1, 3, 5])
sigmoid_values = sigmoid(z_values)

sigmoid_table = pd.DataFrame({
    "z": z_values,
    "Sigmoid(z)": sigmoid_values
})

print(sigmoid_table)

## 14. Plot Sigmoid Function

In [ ]:
z = np.linspace(-10, 10, 200)
probabilities = sigmoid(z)

plt.figure(figsize=(8, 5))
plt.plot(z, probabilities)
plt.axhline(0.5, linestyle="--")
plt.axvline(0, linestyle="--")
plt.xlabel("z")
plt.ylabel("Probability")
plt.title("Sigmoid Function")
plt.grid(True)
plt.show()

## 15. Decision Boundary

The decision boundary separates different classes.

For binary Logistic Regression, the default threshold is usually 0.50.

Probability < 0.50 → Class 0

Probability >= 0.50 → Class 1

In [ ]:
sample_probabilities = np.array([0.20, 0.40, 0.49, 0.50, 0.70, 0.90])

sample_predictions = (sample_probabilities >= 0.50).astype(int)

decision_table = pd.DataFrame({
    "Probability": sample_probabilities,
    "Prediction": sample_predictions
})

print(decision_table)

## 16. Threshold

A threshold converts a probability into a class.

Default threshold = 0.50.

Changing the threshold changes how easily the model predicts Class 1.

In [ ]:
threshold = 0.30

custom_predictions = (y_probability[:, 1] >= threshold).astype(int)

print("Custom threshold:", threshold)
print("First 20 predictions:")
print(custom_predictions[:20])

## 17. Log Loss

Log Loss measures how well the predicted probabilities match the actual classes.

Lower Log Loss is better.

A confident wrong prediction receives a large penalty.

In [ ]:
loss = log_loss(y_test, y_probability)

print("Log Loss:", round(loss, 4))

## 18. Coefficients

Each feature has a coefficient.

A positive coefficient increases the log-odds of Class 1.

A negative coefficient decreases the log-odds of Class 1.

Because the features were standardized, the coefficients are easier to compare in magnitude.

In [ ]:
coefficient_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

coefficient_df["Absolute_Coefficient"] = coefficient_df["Coefficient"].abs()

coefficient_df = coefficient_df.sort_values(
    by="Absolute_Coefficient",
    ascending=False
)

print(coefficient_df.head(10))

## 19. Intercept

The intercept is the bias term of the Logistic Regression model.

The model calculates:

z = intercept + coefficient1*x1 + coefficient2*x2 + ...

In [ ]:
print("Intercept:", model.intercept_[0])

## 20. Model Evaluation

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 21. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

## 22. Multiclass Classification

Multiclass classification means there are three or more possible classes.

Example:
- Class 0 = Setosa
- Class 1 = Versicolor
- Class 2 = Virginica

Logistic Regression can also be used for multiclass classification.

In [ ]:
iris = load_iris()

X_iris = iris.data
y_iris = iris.target

X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    X_iris,
    y_iris,
    test_size=0.20,
    random_state=42,
    stratify=y_iris
)

iris_scaler = StandardScaler()

X_iris_train_scaled = iris_scaler.fit_transform(X_iris_train)
X_iris_test_scaled = iris_scaler.transform(X_iris_test)

multiclass_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

multiclass_model.fit(
    X_iris_train_scaled,
    y_iris_train
)

iris_pred = multiclass_model.predict(X_iris_test_scaled)

iris_accuracy = accuracy_score(y_iris_test, iris_pred)

print("Multiclass Accuracy:", round(iris_accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_iris_test, iris_pred))

## 23. Logistic Regression Workflow

1. Collect data
2. Separate features and target
3. Split into training and testing data
4. Scale features
5. Train Logistic Regression
6. Calculate probabilities
7. Apply threshold
8. Make predictions
9. Evaluate the model

Workflow:

Data → Features/Target → Train/Test Split → Scaling → Logistic Regression → Sigmoid → Probability → Threshold → Class → Evaluation

## 24. Key Terms Summary

- Classification → Predicts categories/classes.
- Binary Classification → Classification with two classes.
- Logistic Regression → Classification algorithm that predicts probabilities.
- Sigmoid Function → Converts model output into a probability between 0 and 1.
- Probability → Likelihood of belonging to a class.
- Decision Boundary → Boundary that separates classes.
- Threshold → Converts probability into a class.
- Log Loss → Measures probability prediction error.
- Coefficient → Represents the effect of a feature on the model's output.
- Intercept → Bias term of the model.
- Multiclass Classification → Classification with three or more classes.